# Midterm · Plan the day

This midterm checks that you **understand** the ideas from Weeks 1–4, not that you remember them. You will read code, write two small functions, test them, judge AI-written code, and explain one decision.

| Part | What you do | Ideas from | Points |
|---|---|---|---|
| A | Read code and explain *why* | Weeks 1–4 | 20 |
| B | Write a rule: `volunteers_needed` | Weeks 1–2 · Principles 1–2 | 20 |
| C | Work with records: `plan_day` | Week 3 · Principle 3 | 25 |
| D | Write tests that catch bugs | Week 4 · Principle 4 | 15 |
| E | Judge AI-written code | all four principles | 10 |
| F | Your decision, and a clean run | Principle 2 | 10 |
| | | **Total** | **100** |

*Campus Food Rescue is fictional. All organisations, people and numbers are invented.*

## Rules

- **Individual work.** 90 minutes.
- **Allowed:** your own notes, the weekly notebooks and the Python Basics notebook.
- **Not allowed:** AI tools, messaging or sharing code with anyone.
- The parts are independent. If you are stuck, move on and come back later.
- After the midterm, you will have a **3-minute oral check**: the instructor points at one line of your `plan_day` and one of your tests, and you explain them. Code you cannot explain does not count as yours.

**Start:** File → Save a copy in Drive, then run the setup cell.

In [ ]:
# @title ▶ Run this cell first (setup) — it prepares the Food Rescue system
# It unpacks the course system that is stored inside this notebook.
# No internet, no installation. Run it again after Colab restarts.
import base64, hashlib, importlib, io, os, sys, zipfile
_KIT_VERSION = "2026.11.1"
_KIT_SHA = "cfeca0a7dadbe71f7f678b9a7eba78697d6a381e80ac3077e1ff3a23d027c9e4"
_KIT_ZIP = "..."

def _prepare_kit():
    data = base64.b64decode(_KIT_ZIP)
    if hashlib.sha256(data).hexdigest() != _KIT_SHA:
        raise RuntimeError("This notebook's course files are damaged. Download a fresh copy.")
    marker = os.path.join("food_rescue_kit", ".kit_version")
    current = open(marker).read().strip() if os.path.exists(marker) else ""
    keep = ("food_rescue_kit/my_settings.py", "food_rescue_kit/my_rules.py")
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        for name in archive.namelist():
            if current == _KIT_VERSION and os.path.exists(name):
                continue                      # already up to date
            if name in keep and os.path.exists(name):
                continue                      # never overwrite your own files
            os.makedirs(os.path.dirname(name), exist_ok=True)
            with open(name, "wb") as target:
                target.write(archive.read(name))
    with open(marker, "w") as file:
        file.write(_KIT_VERSION)
    folder = os.path.abspath("food_rescue_kit")
    if folder not in sys.path:
        sys.path.insert(0, folder)
    for module in [m for m in sys.modules if m == "rescue" or m.startswith("rescue.")]:
        del sys.modules[module]             # load the fresh copy

_prepare_kit()
import rescue as fr
print("✅ Food Rescue system ready (version " + fr.KIT_VERSION + "). You can continue.")


## The situation

The coordinator plans each afternoon's pickups. Volunteers carry the food in **bicycle baskets**. One basket holds about **15 portions**. Before the volunteers leave, the coordinator wants to know three things: how many portions are approved, how many volunteers are needed, and which offers are still waiting for a decision.

Each offer in this midterm is a small record:

In [ ]:
offers = fr.midterm_offers()
for offer in offers:
    print(offer)

## Part A · Read code (20 points, about 15 min)

For each snippet: **first** write what you think it prints, **then** run it, then explain **why** in one or two sentences. Marks are for the explanation, so use the course words (text, number, branch, return, record, ID …).

### A1 · Adding up

In [ ]:
portions = "30"
extra = "12"
print(portions + extra)

✍️ **A1** · It prints … because …  How would you get 42?

### A2 · Boxes

In [ ]:
portions = 50
per_box = 8
print(portions // per_box, portions % per_box)

✍️ **A2** · It prints … because …  What do the two numbers mean for the volunteers?

### A3 · Labels

In [ ]:
def label(portions):
    if portions >= 5:
        return "standard"
    elif portions >= 20:
        return "team"
    return "hold"

print(label(25))

✍️ **A3** · It prints … because …  How would you fix it?

### A4 · Total

In [ ]:
def total(records):
    count = 0
    for record in records:
        count = count + record["portions"]
    print(count)

result = total([{"portions": 10}, {"portions": 5}])
print("Result:", result)

✍️ **A4** · It prints … because …  Change one line so `result` is 15.

### A5 · Stock list

In [ ]:
stock = {}
stock["Bread rolls"] = 30
stock["Rice meals"] = 18
stock["Bread rolls"] = 12
print(len(stock), stock["Bread rolls"])

✍️ **A5** · It prints … because …  Two different bakeries offered bread rolls. What went wrong, and what is the better design?

## Part B · Write a rule (20 points, about 15 min)

One volunteer carries at most `PORTIONS_PER_VOLUNTEER` portions. Write `volunteers_needed(portions)` that **returns** how many volunteers are needed to carry all the portions.

- You cannot send part of a volunteer: 16 portions need **2** volunteers when one carries 15.
- 0 portions need 0 volunteers.
- Use the constant, not the number 15.

💡 Remember Week 1: `//` gives the full loads, `%` gives what is left over.

In [ ]:
PORTIONS_PER_VOLUNTEER = 15   # Part F: you may change this decision later

def volunteers_needed(portions):
    """Return the number of volunteers needed to carry this many portions."""
    # TODO: use // and %, and an if
    return None

print(volunteers_needed(16))

In [ ]:
fr.check("volunteers_needed", volunteers_needed, {"PORTIONS_PER_VOLUNTEER": PORTIONS_PER_VOLUNTEER})

## Part C · Work with records (25 points, about 20 min)

Write `plan_day(offers)`. It receives a list of offer records like the ones above and **returns** a dictionary with four keys:

| Key | Value |
|---|---|
| `"offers"` | how many offers are in the list |
| `"approved_portions"` | the total portions of **approved** offers only |
| `"volunteers"` | the total volunteers for the approved offers: call your `volunteers_needed` for each one |
| `"waiting"` | a list of the **IDs** of offers that are not approved, in the same order |

Its **contract** also promises: *plan_day never changes the list it is given.* An empty list gives `{"offers": 0, "approved_portions": 0, "volunteers": 0, "waiting": []}`.

In [ ]:
def plan_day(offers):
    """Return a summary of the day: offers, approved_portions, volunteers, waiting."""
    approved_portions = 0
    volunteers = 0
    waiting = []
    # TODO: loop over the offers and update the three values

    return {"offers": 0,  # TODO
            "approved_portions": approved_portions,
            "volunteers": volunteers,
            "waiting": waiting}

print(plan_day(offers))

The check below uses a correct `volunteers_needed` while it checks `plan_day`, so a mistake in Part B does not cost you marks twice.

In [ ]:
fr.check("plan_day", plan_day, {"PORTIONS_PER_VOLUNTEER": PORTIONS_PER_VOLUNTEER})

## Part D · Tests that catch bugs (15 points, about 15 min)

A good test **fails when the code is wrong**. Write four tests with `assert`:

1. `test_normal`: `volunteers_needed` with a normal amount that is **not** an exact full load.
2. `test_boundary`: `volunteers_needed` with **exactly** one full load (and one portion more).
3. `test_waiting`: `plan_day` with a list that contains an offer that is **not approved**.
4. `test_empty`: `plan_day` with an empty list.

💡 Write amounts **with the constant**, for example `volunteers_needed(PORTIONS_PER_VOLUNTEER + 1) == 2`, so your tests still work if the decision changes (Part F).

In [ ]:
def test_normal():
    # TODO
    pass

def test_boundary():
    # TODO
    pass

def test_waiting():
    # TODO
    pass

def test_empty():
    # TODO
    pass

Now run your tests. `fr.catch_bugs` runs them on **your** code, then on four versions with a **planted bug**, and finally with a different decision. Each bug your tests catch shows that the tests protect the system.

In [ ]:
fr.catch_bugs([test_normal, test_boundary, test_waiting, test_empty])

## Part E · Judge AI-written code (10 points, about 10 min)

A classmate asked an AI assistant to write `plan_day`. This is the answer. It runs without an error. A new offer has just arrived (F005); nobody has decided about it yet, so it has no `"approved"` value. Run the cell, then look closely.

In [ ]:
def plan(x):
    t = 0
    v = 0
    w = []
    for o in x:
        if o.get("approved", True):
            t = t + o["portions"]
            v = v + o["portions"] // 15 + 1
        else:
            w.append(o["food"])
    return "Approved: " + str(t) + " portions, " + str(v) + " volunteers. Waiting: " + ", ".join(w)

today = fr.midterm_offers() + [{"id": "F005", "food": "Bananas", "portions": 20}]
result = plan(today)
print(result)
print("Type of the result:", type(result).__name__)

✍️ Find **at least three** problems. For each one, name the principle and say which offer or person would be harmed.

| Problem (point at the line) | Principle | Who is harmed, and how |
|---|---|---|
| | | |
| | | |
| | | |

🤔 Only F001, F002 and F004 are approved, so the total should be 60 portions and 5 volunteers. Why does the AI version say 80 and 8? And which bananas are "waiting"?

## Part F · Your decision, and a clean run (10 points, about 10 min)

The default assumes bicycle baskets (15 portions). Your campus may be different: volunteers on foot might carry 8, a motorbike might carry 30.

1. Choose your `PORTIONS_PER_VOLUNTEER` and change it in the **Part B** cell. You may keep 15 if you can explain why.
2. **Runtime → Restart session and run all.** Do Parts B, C and D still pass? In Part D, step 3 tells you if a raw number is hidden in your code or tests.
3. If a test failed only because the decision changed, rewrite it with the constant.

In [ ]:
print("My decision:", PORTIONS_PER_VOLUNTEER, "portions per volunteer")
print("Today's plan:", plan_day(fr.midterm_offers()))

📝 **Decision log · Midterm**

| | |
|---|---|
| **My choice** | … portions per volunteer |
| **An alternative I considered** | |
| **Why I chose it** | |
| **Who is affected (the tradeoff)** | |
| **What happened when I changed it and ran everything again** | |

## Submit

- [ ] Part A: five explanations, each with a *why*.
- [ ] Parts B and C: checks run (✅ or not, show your best attempt).
- [ ] Part D: `catch_bugs` output is visible.
- [ ] Part E: the table has at least three problems.
- [ ] Part F: decision log filled in, and the notebook was **restarted and run all** before saving.

Download your notebook (**File → Download → .ipynb**) and submit it as your instructor announces. Then prepare for the 3-minute oral check: be ready to explain any line of your `plan_day` and one of your tests.